# **SQL Queries**

**1. DATA LOADING**

In [22]:
import pandas as pd
import sqlite3
import re

file_path = '/content/anime_catalog.csv'
raw_df = pd.read_csv(file_path)

**2. DATABASE SETUP (SQLite in-memory)**

Creating tables according to relational schema (Many-to-Many logic)

In [23]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript("""
-- 1. Reference Tables (Dictionaries)
CREATE TABLE genres (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE);
CREATE TABLE studios (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE);
CREATE TABLE directors (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE);
CREATE TABLE dubbing_groups (id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT UNIQUE);

-- 2. Main Anime Table
CREATE TABLE anime (
    anime_id INT PRIMARY KEY,
    title_ru TEXT,
    alt_names TEXT,
    anime_type TEXT,
    status TEXT,
    release_year INT,
    age_rating TEXT,
    source TEXT,
    url TEXT
);

-- 3. Statistics Table (1:1 Relationship with Anime)
CREATE TABLE anime_stats (
    anime_id INT PRIMARY KEY REFERENCES anime(anime_id) ON DELETE CASCADE,
    site_rating DECIMAL,
    site_votes INT,
    site_views INT,
    shikimori_rating DECIMAL,
    mal_rating DECIMAL,
    kinopoisk_rating DECIMAL,
    comments_count INT,
    reviews_count INT
);

-- 4. Junction Tables (Many-to-Many Relationships)
CREATE TABLE anime_genres (
    anime_id INT REFERENCES anime(anime_id),
    genre_id INT REFERENCES genres(id),
    PRIMARY KEY (anime_id, genre_id)
);
CREATE TABLE anime_studios (
    anime_id INT REFERENCES anime(anime_id),
    studio_id INT REFERENCES studios(id),
    PRIMARY KEY (anime_id, studio_id)
);
CREATE TABLE anime_directors (
    anime_id INT REFERENCES anime(anime_id),
    director_id INT REFERENCES directors(id),
    PRIMARY KEY (anime_id, director_id)
);
""")


**3. CLEANING FUNCTIONS & MIGRATION**

In [24]:
def to_int(val):
    if pd.isna(val) or val == 'N/A': return 0
    num = re.sub(r'\D', '', str(val))
    return int(num) if num else 0

def to_float(val):
    if pd.isna(val) or val == 'N/A': return 0.0
    try: return float(val)
    except: return 0.0

def link_entity(anime_id, column_val, table_name, link_table, link_col_name):
    """Helper function to populate dictionaries and link tables"""
    if pd.isna(column_val) or column_val == 'N/A': return

    entities = [e.strip() for e in str(column_val).split('|')]
    for name in entities:
        cursor.execute(f"INSERT OR IGNORE INTO {table_name} (name) VALUES (?)", (name,))
        entity_id = cursor.execute(f"SELECT id FROM {table_name} WHERE name = ?", (name,)).fetchone()[0]
        cursor.execute(f"INSERT OR IGNORE INTO {link_table} (anime_id, {link_col_name}) VALUES (?, ?)", (anime_id, entity_id))

for _, row in raw_df.iterrows():

    year_match = re.search(r'(\d{4})', str(row['year']))
    year = int(year_match.group(1)) if year_match else None

    cursor.execute("""
        INSERT INTO anime (anime_id, title_ru, alt_names, anime_type, status, release_year, age_rating, source, url)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (row['anime_id'], row['title_ru'], row['alt_names'], row['type'], row['status'], year, row['age_rating'], row['source'], row['url']))

    cursor.execute("""
        INSERT INTO anime_stats (anime_id, site_rating, site_votes, site_views, shikimori_rating, mal_rating, kinopoisk_rating, comments_count, reviews_count)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (row['anime_id'], to_float(row['site_rating']), row['site_votes'], to_int(row['site_views']),
          to_float(row['shikimori_rating']), to_float(row['mal_rating']), to_float(row['kinopoisk_rating']),
          to_int(row['comments_count']), to_int(row['reviews_count'])))

    link_entity(row['anime_id'], row['genres'], 'genres', 'anime_genres', 'genre_id')
    link_entity(row['anime_id'], row['studio'], 'studios', 'anime_studios', 'studio_id')
    link_entity(row['anime_id'], row['director'], 'directors', 'anime_directors', 'director_id')

conn.commit()
print("Success: Data migrated to relational structure.")


Success: Data migrated to relational structure.


**4. DATA ANALYSIS (SQL QUERIES)**

In [25]:
def run_query(title, description, sql):
    print(f"--- Query: {title} ---")
    print(f"Insight: {description}")
    display(pd.read_sql(sql, conn))
    print("\n")

Query 1: COUNT + GROUP BY + ORDER BY

In [26]:
run_query(
    "Distribution by Format",
    "Counts the number of anime in each category (ONA, TV, Movie) to understand catalog variety.",
    "SELECT anime_type, COUNT(*) as total FROM anime GROUP BY anime_type ORDER BY total DESC"
)

--- Query: Distribution by Format ---
Insight: Counts the number of anime in each category (ONA, TV, Movie) to understand catalog variety.


,anime_type,total
0,ONA,11
1,Сериал,7
2,Полнометражный фильм,3
3,Неизвестно,3


 Query 2: AVG + JOIN + GROUP BY

In [27]:
run_query(
    "Average Rating by Age Category",
    "Calculates the average user rating for different age ratings to see target audience satisfaction.",
    """
    SELECT a.age_rating, ROUND(AVG(s.site_rating), 2) as avg_score
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    WHERE a.age_rating != 'Unknown' AND a.age_rating != 'Неизвестно'
    GROUP BY a.age_rating
    ORDER BY avg_score DESC
    """
)

--- Query: Average Rating by Age Category ---
Insight: Calculates the average user rating for different age ratings to see target audience satisfaction.


,age_rating,avg_score
0,PG,8.64
1,PG-13,1.07
2,R-17+,0.00
3,G,0.00


Query 3: SUM + JOIN + GROUP BY

In [28]:
run_query(
    "Genre Popularity by Views",
    "Sum of site views per genre, calculated through the junction table.",
    """
    SELECT g.name as genre, SUM(s.site_views) as total_views
    FROM genres g
    JOIN anime_genres ag ON g.id = ag.genre_id
    JOIN anime_stats s ON ag.anime_id = s.anime_id
    GROUP BY g.name
    ORDER BY total_views DESC
    LIMIT 10
    """
)


--- Query: Genre Popularity by Views ---
Insight: Sum of site views per genre, calculated through the junction table.


,genre,total_views
0,Фэнтези,74069
1,Не японское,70188
2,Приключения,67837
3,Боевые искусства,56166
4,Магия,52478
5,Экшен,26637
6,Романтика,13033
7,Китайское 3D,11597
8,Школьная жизнь,8927
9,Комедия,7843


Query 4: WHERE + ORDER BY (Filtering)

In [29]:
run_query(
    "Most Anticipated Upcoming Releases (2026-2027)",
    "Filters future projects and sorts them by user votes/interest.",
    """
    SELECT a.title_ru, a.release_year, s.site_votes, a.status
    FROM anime a
    JOIN anime_stats s ON a.anime_id = s.anime_id
    WHERE a.release_year >= 2026
    ORDER BY s.site_votes DESC
    LIMIT 5
    """
)

--- Query: Most Anticipated Upcoming Releases (2026-2027) ---
Insight: Filters future projects and sorts them by user votes/interest.


,title_ru,release_year,site_votes,status
0,Легенда об Аанге: Последний маг воздуха,2026,98,вышел
1,Расцвет молодости: Наша весна,2026,9,онгоинг
2,Непревзойдённый под небесами: Дрейфующий ледян...,2027,0,анонс
3,Глупая игра богов,2027,0,анонс
4,Сиротан,2026,0,анонс


Query 5: Complex JOIN (Studio Performance)

In [30]:
run_query(
    "Studio Reach and Efficiency",
    "Total view count and project count per studio to measure market presence.",
    """
    SELECT st.name as studio, COUNT(ast.anime_id) as project_count, SUM(s.site_views) as total_views
    FROM studios st
    JOIN anime_studios ast ON st.id = ast.studio_id
    JOIN anime_stats s ON ast.anime_id = s.anime_id
    GROUP BY st.name
    HAVING total_views > 0
    ORDER BY total_views DESC
    """
)

--- Query: Studio Reach and Efficiency ---
Insight: Total view count and project count per studio to measure market presence.


,studio,project_count,total_views
0,Avatar Studios,1,52478
1,CG Year,2,4099
2,Yanchester,1,3882
3,Fugaku,1,3444
4,Shengying,2,2862
5,ASK Animation,1,1836
6,Outline,1,1642
7,Shenman Entertainment,1,1373
8,CloverWorks,1,1360
9,Xing Yi Kai Chen,1,1177
